## Thread-safe vs Non-thread-safe
This is important in LLD because we often design classes whose objects are accessed by multiple threads.

In [1]:
# non Thread safe
class BankAccount:
    def __init__(self):
        self.balance = 1000

    def withdraw(self, amount):
        if self.balance >= amount:
            self.balance -= amount
            return True
        return False

Now imagine two threads withdrawing ₹700 simultaneously.

Both might read the balance as ₹1000 before either updates it.

Both withdrawals could succeed, even though the account only has ₹1000.

This is a race condition.

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 1000
        self.lock = threading.Lock()

    def withdraw(self, amount):
        with self.lock:
            if self.balance >= amount:
                self.balance -= amount
                return True
            return False

In [2]:
import threading

account = BankAccount()

def withdraw_money():
    result = account.withdraw(700)
    print(threading.current_thread().name, result)

t1 = threading.Thread(target=withdraw_money, name="T1")
t2 = threading.Thread(target=withdraw_money, name="T2")

t1.start()
t2.start()

t1.join()
t2.join()

print("Final balance:", account.balance)

T1 True
T2 False
Final balance: 300


## Normal py list 

In [3]:
tasks = []

def add_task(task):
    tasks.append(task)

def get_task():
    if tasks:
        return tasks.pop(0)

If multiple threads access this collection, individual list operations may be safe in some CPython implementations, but the combined check-and-remove operation isn't guaranteed to be atomic.

For example:
1. Thread A checks if tasks — list is not empty.
2. Thread B removes the last task.
3. Thread A tries pop(0) — gets an IndexError.
This is a race condition.

In [5]:
from queue import Queue
tasks = Queue()
tasks.put("Task 1")
tasks.put("Task 2")
task = tasks.get()
task=tasks.get()
print(task)

Task 2


put() adds an item safely.



get() retrieves and removes an item safely.



Multiple threads can use the same queue without you manually implementing locks.

In [9]:
import threading
from queue import Queue
import time

tasks = Queue()

# Producer: adds tasks
def producer():
    for i in range(1, 6):
        task = f"Task {i}"
        tasks.put(task)
        print(f"Produced: {task}")

# Consumer: processes tasks
def consumer():
    while True:
        task = tasks.get()

        if task is None:
            tasks.task_done()
            break

        print(f"{threading.current_thread().name} processing {task}")
        time.sleep(1)

        tasks.task_done()


# Start producer
producer_thread = threading.Thread(target=producer)
producer_thread.start()

# Start two consumers
worker1 = threading.Thread(target=consumer, name="Worker-1")
worker2 = threading.Thread(target=consumer, name="Worker-2")

worker1.start()
worker2.start()

# Wait for all tasks to be produced
producer_thread.join()

# Send shutdown signals
tasks.put(None)
tasks.put(None)

# Wait until all tasks are completed
tasks.join()

worker1.join()
worker2.join()

print("All tasks completed")

Produced: Task 1
Produced: Task 2
Produced: Task 3
Produced: Task 4
Produced: Task 5
Worker-1 processing Task 1
Worker-2 processing Task 2
Worker-2 processing Task 3
Worker-1 processing Task 4
Worker-2 processing Task 5
All tasks completed
